In [2]:
import pandas as pd
import re


df = pd.read_csv("../Scrape/DoAn_HUST_Chrome.csv", encoding="utf-8-sig")

df.drop_duplicates(inplace=True)

cols_to_clean = ["Order", "GiangVien", "TenDeTai", "ChiTiet", "LoaiDoAn"]

for col in cols_to_clean:
    df[col] = df[col].astype(str).str.replace(r'\n|\t', ' ', regex=True).str.strip()
    
print("fagdfsafd")
print(df.head())

fagdfsafd
  Order          GiangVien                                           TenDeTai  \
0   nan                nan                                                nan   
1   2.0  Nguyễn Tiến Thành                                       Anti Scammer   
2   nan                nan   Hệ thống tự sinh câu hỏi trắc nghiệm từ hình ảnh   
3   nan                nan  Optimal Multi-agent Path Finding for Game Dead...   
4   nan                nan  Optimal Multi-agent Path Finding for Age of Em...   

                                             ChiTiet  \
0                                                nan   
1  https://docs.google.com/presentation/d/1wGGbne...   
2  Hệ thống đã được xây dựng một phần, xem video ...   
3  Có nhiều game đã thực hiện việc tìm đường cho ...   
4  chi tiết có đọc ở đây https://docs.google.com/...   

                                LoaiDoAn  
0                                    nan  
1  TT ,, ĐATN ,, TTTN ,, ĐAMH ,, SV NCKH  
2  TT ,, ĐATN ,, TTTN ,, ĐAMH ,, SV N

In [3]:
print(df.info())
print(df.describe())


<class 'pandas.core.frame.DataFrame'>
Index: 482 entries, 0 to 486
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Order      482 non-null    object
 1   GiangVien  482 non-null    object
 2   TenDeTai   482 non-null    object
 3   ChiTiet    482 non-null    object
 4   LoaiDoAn   482 non-null    object
dtypes: object(5)
memory usage: 22.6+ KB
None
       Order GiangVien                             TenDeTai ChiTiet LoaiDoAn
count    482       482                                  482     482      482
unique    59        59                                  481     436       17
top      nan       nan  Xây dựng hệ thống đánh giá tín dụng     nan      nan
freq     424       424                                    2      46      208


In [4]:
df.drop(df.index[0], inplace=True)

In [ ]:
import pandas as pd
import numpy as np

# 1. Giả lập dữ liệu để minh họa (bỏ qua bước này nếu bạn đã có df)
# df = pd.read_csv("DoAn_HUST_Chrome.csv")

# 2. Xử lý khoảng trắng thành NaN (để chắc chắn ffill hoạt động)
# Regex này bắt các chuỗi rỗng hoặc chỉ chứa dấu cách
# df['GiangVien'] = df['GiangVien'].replace(r'^\s*$', np.nan, regex=True)
df['GiangVien'] = df['GiangVien'].replace('nan', np.nan, regex=True)

# 3. Thực hiện Forward Fill (lấy giá trị gần nhất phía trên)
df['GiangVien'] = df['GiangVien'].ffill()
print(df['GiangVien'].ffill().head(10))
print(type(df['GiangVien'][2]), df['GiangVien'][20])
# Kiểm tra lại
print(df[['Order', 'GiangVien', 'TenDeTai']].head(10))

1     Nguyễn Tiến Thành
2     Nguyễn Tiến Thành
3     Nguyễn Tiến Thành
4     Nguyễn Tiến Thành
5     Nguyễn Tiến Thành
6     Nguyễn Tiến Thành
7     Nguyễn Tiến Thành
8     Nguyễn Tiến Thành
9     Nguyễn Tiến Thành
10    Nguyễn Tiến Thành
Name: GiangVien, dtype: object
<class 'str'> Nguyễn Tiến Thành
   Order          GiangVien                                           TenDeTai
1    2.0  Nguyễn Tiến Thành                                       Anti Scammer
2    nan  Nguyễn Tiến Thành   Hệ thống tự sinh câu hỏi trắc nghiệm từ hình ảnh
3    nan  Nguyễn Tiến Thành  Optimal Multi-agent Path Finding for Game Dead...
4    nan  Nguyễn Tiến Thành  Optimal Multi-agent Path Finding for Age of Em...
5    nan  Nguyễn Tiến Thành          Game ném đĩa cross platform multi-players
6    nan  Nguyễn Tiến Thành  Ứng dụng DeepFake trong việc tạo video bố mẹ t...
7    nan  Nguyễn Tiến Thành  Xây dựng phần mềm giả lập việc định tuyến cho ...
8    nan  Nguyễn Tiến Thành  Mô phỏng biến đổi cảm xúc của con ng

In [17]:
print(type(df['GiangVien'][2]), df['GiangVien'][30])

<class 'str'> Trịnh Thành Trung


In [18]:
def create_rag_document(row):
    """
    Hàm biến 1 dòng DataFrame thành 1 đoạn văn bản ngữ nghĩa.
    """
    # 1. Làm sạch dữ liệu text cơ bản
    topic = str(row['TenDeTai']).strip()
    teacher = str(row['GiangVien']).strip()
    detail = str(row['ChiTiet']).strip()
    proj_type = str(row['LoaiDoAn']).strip()
    
    # 2. Tạo nội dung Semantic (Dùng để Embed)
    # Kỹ thuật: Đưa các keyword quan trọng (Tên đề tài, Giảng viên) lên đầu
    content = f"""
    Tên đề tài: {topic}
    Giảng viên hướng dẫn: {teacher}
    Mô tả chi tiết: {detail}
    Loại đồ án: {proj_type}
    """.strip()
    
    # 3. Tạo Metadata (Dùng để lọc sau này, ví dụ: "Tìm đề tài của thầy Thành")
    metadata = {
        "teacher_name": teacher,
        "project_type": proj_type,
        "topic_name": topic # Lưu lại để trích xuất nhanh nếu cần
    }
    
    return content, metadata

# Áp dụng vào DataFrame
rag_data = []

for index, row in df.iterrows():
    content, meta = create_rag_document(row)
    rag_data.append({
        "id": str(index),       # ID duy nhất
        "text": content,        # Nội dung để embed
        "metadata": meta        # Metadata để filter
    })

# Xem thử kết quả của 1 mẫu
print("--- MẪU DỮ LIỆU CHO RAG ---")
print("Text chunk:\n", rag_data[0]['text'])
print("\nMetadata:\n", rag_data[0]['metadata'])

--- MẪU DỮ LIỆU CHO RAG ---
Text chunk:
 Tên đề tài: Anti Scammer
    Giảng viên hướng dẫn: Nguyễn Tiến Thành
    Mô tả chi tiết: https://docs.google.com/presentation/d/1wGGbneUOg8jJQ8Y6Wt7-nlCYhzQ8nKuN9hpKaMM3wYU/edit?usp=sharing
    Loại đồ án: TT ,, ĐATN ,, TTTN ,, ĐAMH ,, SV NCKH

Metadata:
 {'teacher_name': 'Nguyễn Tiến Thành', 'project_type': 'TT ,, ĐATN ,, TTTN ,, ĐAMH ,, SV NCKH', 'topic_name': 'Anti Scammer'}


In [ ]:
import pandas as pd
import numpy as np
from langchain_core.documents import Document

# 1. Đọc và xử lý dữ liệu (như bạn đã làm)
df = pd.read_csv("DoAn_HUST_Chrome.csv")
# Fill dữ liệu thiếu cho cột GiangVien (Forward Fill)
df['GiangVien'] = df['GiangVien'].replace(r'^\s*$', np.nan, regex=True).ffill()

# 2. Chuyển đổi thành LangChain Documents
documents = []
for index, row in df.iterrows():
    # Tạo nội dung text để Search (Tiếng Việt)
    content = f"""
    Tên đề tài: {row['TenDeTai']}
    Giảng viên: {row['GiangVien']}
    Loại đồ án: {row['LoaiDoAn']}
    Chi tiết: {row['ChiTiet']}
    """.strip()
    
    # Tạo Metadata để lọc
    metadata = {
        "source_id": str(index),
        "teacher": str(row['GiangVien']),
        "type": str(row['LoaiDoAn'])
    }
    
    # Tạo object Document
    doc = Document(page_content=content, metadata=metadata)
    documents.append(doc)

print(f"Đã chuẩn bị xong {len(documents)} văn bản.")

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Khởi tạo Embedding Model (Chạy local, lần đầu sẽ tải model về máy khoảng 500MB)
print("Đang tải model Embedding (chỉ lần đầu)...")
embedding_model = HuggingFaceEmbeddings(model_name="keepitreal/vietnamese-sbert")
# Hoặc dùng model đa ngôn ngữ nhẹ hơn: "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# 2. Khởi tạo ChromaDB và nạp dữ liệu
# persist_directory: Đường dẫn thư mục để lưu vector (để lần sau không phải chạy lại)
persist_dir = "./chroma_db_hust"

print("Đang tạo Vector DB...")
vector_db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=persist_dir
)

print("✅ Đã lưu Vector DB thành công!")

In [ ]:
import os
import pandas as pd
import numpy as np
import getpass
from dotenv import load_dotenv
load_dotenv()

# Import các thư viện LangChain & Google
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# --- 1. CẤU HÌNH API KEY ---
# Bạn có thể set cứng hoặc nhập khi chạy (an toàn hơn)
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Nhập Google API Key của bạn: ")

# --- 2. XỬ LÝ DỮ LIỆU (Giống logic trước) ---
print("📊 Đang xử lý dữ liệu...")
df = pd.read_csv("DoAn_HUST_Chrome.csv") # Đổi tên file nếu cần

# Fill dữ liệu thiếu

# Tạo LangChain Documents
documents = []
for index, row in df.iterrows():
    # Nội dung ngữ nghĩa để Embedding
    content = f"""
    Tên đề tài: {row['TenDeTai']}
    Giảng viên hướng dẫn: {row['GiangVien']}
    Loại đồ án: {row['LoaiDoAn']}
    Chi tiết mô tả: {row['ChiTiet']}
    """.strip()
    
    # Metadata để lọc
    metadata = {
        "source_id": str(index),
        "teacher": str(row['GiangVien']),
        "type": str(row['LoaiDoAn'])
    }
    
    documents.append(Document(page_content=content, metadata=metadata))

print(f"✅ Đã chuẩn bị {len(documents)} văn bản.")

# --- 3. KHỞI TẠO GOOGLE EMBEDDING & VECTOR DB ---
# Sử dụng model embedding mới nhất của Google hỗ trợ đa ngôn ngữ tốt
embedding_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

persist_dir = "./chroma_db_gemini"

# Kiểm tra xem DB đã tồn tại chưa để tránh embed lại tốn thời gian
if os.path.exists(persist_dir):
    print("🔄 Đã tìm thấy Vector DB cũ, đang tải lên...")
    vector_db = Chroma(persist_directory=persist_dir, embedding_function=embedding_model)
else:
    print("🚀 Đang tạo Vector DB mới (đang gửi dữ liệu lên Google để embed)...")
    vector_db = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_dir
    )
    print("✅ Đã tạo xong Vector DB!")

# --- 4. KHỞI TẠO LLM (GEMINI 1.5 FLASH) ---
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    temperature=0, # 0 để trả lời chính xác, ít "chém gió"
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

# --- 5. TẠO RAG CHAIN ---
# Prompt template
system_prompt = (
    "Bạn là trợ lý AI thông minh của sinh viên Bách Khoa."
    "Dựa vào danh sách đề tài đồ án được cung cấp dưới đây để trả lời câu hỏi."
    "Nếu không tìm thấy thông tin trong ngữ cảnh, hãy nói rõ là không có dữ liệu."
    "Hãy trả lời ngắn gọn, súc tích và định dạng đẹp (dùng gạch đầu dòng)."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Tạo Chain
retriever = vector_db.as_retriever(search_kwargs={"k": 5}) # Lấy 5 đề tài liên quan nhất
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# --- 6. CHẠY THỬ ---
while True:
    query = input("\n🔍 Nhập câu hỏi (hoặc 'exit' để thoát): ")
    if query.lower() == 'exit':
        break
    
    print("⏳ Gemini đang suy nghĩ...")
    try:
        response = rag_chain.invoke({"input": query})
        print("\n🤖 GEMINI TRẢ LỜI:")
        print(response["answer"])
        
        # (Tuỳ chọn) Xem nó đã lấy thông tin từ đề tài nào
        # print("\n[Nguồn tham khảo]:")
        # for doc in response["context"]:
        #     print(f"- {doc.page_content.splitlines()[0]}")
            
    except Exception as e:
        print(f"❌ Lỗi: {e}")

In [3]:
from langchain_core.output_parsers import StrOutputParser

d:\code language\anaconda3\envs\LLM_RAG\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_core.runnables import RunnablePassthrough

In [6]:
# check_embedding_models.py
!pip install google-generative-ai --upgrade

import os
import google.generativeai as genai

# --- CẤU HÌNH API KEY ---
# Bạn có thể set biến môi trường hoặc điền trực tiếp vào đây (không khuyến khích hardcode)
# os.environ["GOOGLE_API_KEY"] = "AIzaSy..." 

if not os.getenv("GOOGLE_API_KEY"):
    print("❌ Lỗi: Chưa tìm thấy biến môi trường GOOGLE_API_KEY.")
    print("Vui lòng chạy: export GOOGLE_API_KEY='API_KEY_CUA_BAN' trước khi chạy script.")
    exit(1)

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

def list_embedding_models():
    print(f"{'='*50}")
    print("DANG KIEM TRA CAC MODEL HO TRO EMBEDDING...")
    print(f"{'='*50}")
    
    found_any = False
    try:
        # Lấy danh sách tất cả model
        for m in genai.list_models():
            # Kiểm tra xem model có hỗ trợ phương thức 'embedContent' không
            if 'embedContent' in m.supported_generation_methods:
                print(f"✅ Tên model: {m.name}")
                print(f"   - Display Name: {m.display_name}")
                print(f"   - Input Token Limit: {m.input_token_limit}")
                print(f"   ---")
                found_any = True
        
        if not found_any:
            print("⚠️ Không tìm thấy model nào hỗ trợ 'embedContent' với API Key này.")
    
    except Exception as e:
        print(f"❌ Có lỗi xảy ra khi kết nối tới Google API: {e}")

if __name__ == "__main__":
    list_embedding_models()

ERROR: Could not find a version that satisfies the requirement google-generative-ai (from versions: none)
ERROR: No matching distribution found for google-generative-ai


ModuleNotFoundError: No module named 'google.generativeai'